In [ ]:
%load_ext sql

In [ ]:
%sql spark

##### Delete Existing Delta Table

In [ ]:
%%sql

DROP TABLE IF EXISTS delta.`s3a://warehouse/default/deltalake/invoices_ttv`;

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/invoices_ttv --recursive

##### Create Delta Table

In [ ]:
df = spark.read.format("parquet").load("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
df.printSchema()

#
display(df)

# Save as delta table into Minio S3
df.write.format('delta').save('s3a://warehouse/default/deltalake/invoices_ttv')

In [ ]:
%%sql

CREATE OR REPLACE TABLE delta.`s3a://warehouse/default/deltalake/invoices_ttv` USING  DELTA
AS SELECT * FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`;

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_ttv`
LIMIT 5;

In [ ]:
%%sql

DELETE FROM delta.`s3a://warehouse/default/deltalake/invoices_ttv`
WHERE customer_id = 1;

In [ ]:
%%sql

UPDATE delta.`s3a://warehouse/default/deltalake/invoices_ttv`
SET quantity = 25
WHERE customer_id = 5; 

In [ ]:
%%sql

INSERT INTO delta.`s3a://warehouse/default/deltalake/invoices_ttv`
SELECT * FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`

In [ ]:
%%sql


SELECT COUNT(*) FROM delta.`s3a://warehouse/default/deltalake/invoices_ttv`
WHERE customer_id > 100; 

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://warehouse/default/deltalake/invoices_ttv`;

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_ttv` VERSION AS OF 0
WHERE customer_id = 1;

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_ttv` TIMESTAMP AS OF '2026-03-31T07:45:09.000+00:00'
WHERE customer_id = 1;

In [ ]:
%%sql

RESTORE TABLE delta.`s3a://warehouse/default/deltalake/invoices_ttv` TO VERSION AS OF 0; 
-- RESTORE TABLE delta.`s3://warehouse/default/deltalake/invoices_ttv` TO TIMESTAMP AS OF '2025-02-02T07:08:09.000+00:00';

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_ttv`
WHERE customer_id = 1;

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://warehouse/default/deltalake/invoices_ttv`;

In [0]:
-- 3	2025-02-02T07:09:57.000+00:00
--   2025-02-02T07:09:35.000+00:00
-- 2 2025-02-02T07:09:31.000+00:00

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_ttv` TIMESTAMP AS OF '2026-03-31 13:19:57' -- version 2
WHERE customer_id = 5;  

In [ ]:
%%sql

SELECT * FROM delta.`s3a://warehouse/default/deltalake/invoices_ttv` -- version 0
WHERE customer_id = 5;  

In [ ]:

from pyspark.sql.functions import col
df = spark.read.option("versionAsOf", "1").table("delta.`s3a://warehouse/default/deltalake/invoices_ttv`")
display(df.filter(col("customer_id") == 1))

In [ ]:

df = spark.read.option("versionAsOf", "2").table("delta.`s3a://warehouse/default/deltalake/invoices_ttv`")
display(df.filter(col("customer_id") == 5))

In [ ]:

df = spark.read.option("timestampAsOf", "2026-03-31 13:19:57").table("delta.`s3a://warehouse/default/deltalake/invoices_ttv`")
display(df.filter(col("customer_id") == 5))